In [1]:
#imports
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql.functions import lit, monotonically_increasing_id
import config.ConnectionConfig as cc

In [2]:
cc.setupEnvironment()
spark = cc.startLocalCluster("DIM_LOCK", 4)
spark.getActiveSession()

25/05/09 12:42:15 WARN Utils: Your hostname, 4L3KS-comp resolves to a loopback address: 127.0.1.1; using 10.140.99.119 instead (on interface wlp2s0)
25/05/09 12:42:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/aleks/Downloads/bigtools/spark-3.5.4-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/aleks/.ivy2/cache
The jars for the packages stored in: /home/aleks/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.postgresql#postgresql added as a dependency
org.elasticsearch#elasticsearch-spark-30_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-09e6402c-e91f-4788-b91d-311878ca7288;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.0 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.9.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.

# Extract

In [3]:
cc.set_connectionProfile("default")

df_locks = spark.read.format("jdbc")\
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "locks") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "lockid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 10000) \
    .load()

df_stations = spark.read.format("jdbc")\
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "stations") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "stationid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 10000) \
    .load()

# Transform

In [4]:
#use a spark implementation, instead of monotically_incresing_id, cause this bitch doesn't increase consecutively
#nvm , apparently we can use monotically_increasing_id()
df_dim_lock = df_locks.join(df_stations, df_locks.stationid == df_stations.stationid, "left")\
    .select(
        df_locks.lockid.alias("lock_id"),
        df_stations.stationid.alias("station_id"),
        df_stations.stationnr.alias("station_nr"),
        df_stations.street,
        df_stations.number,
        df_stations.zipcode,
        df_stations.district,
        df_stations.gpscoord.alias("gps_coord")
    )
#DK how to do the no_lock thingy

# LOAD

In [5]:
#no lock thingy
columns = ["lock_id", "station_id", "stationnr", "street", "number", "zipcode", "district","gps_coord"]
df_no_lock = spark.createDataFrame([('9999', '9999', 'None', 'None', 'None', 'None','None', '0,0')], columns)
df_updated = df_no_lock.union(df_dim_lock)
df_updated.show()

+-------+----------+---------+-----------+------+-------+---------+-----------------+
|lock_id|station_id|stationnr|     street|number|zipcode| district|        gps_coord|
+-------+----------+---------+-----------+------+-------+---------+-----------------+
|   9999|      9999|     None|       None|  None|   None|     None|              0,0|
|     19|         2|      019| ONTBREKEND|    12|   2000|ANTWERPEN| (51.219,4.40405)|
|     20|         2|      019| ONTBREKEND|    12|   2000|ANTWERPEN| (51.219,4.40405)|
|     21|         2|      019| ONTBREKEND|    12|   2000|ANTWERPEN| (51.219,4.40405)|
|      1|         1|      026|Meir (2000)|    84|   2000|ANTWERPEN|(51.2182,4.41241)|
|      2|         1|      026|Meir (2000)|    84|   2000|ANTWERPEN|(51.2182,4.41241)|
|      3|         1|      026|Meir (2000)|    84|   2000|ANTWERPEN|(51.2182,4.41241)|
|      4|         1|      026|Meir (2000)|    84|   2000|ANTWERPEN|(51.2182,4.41241)|
|      5|         1|      026|Meir (2000)|    84|   20

In [6]:
df_updated.write.format("delta").mode("overwrite").saveAsTable("dimLock")
df_updated.repartition(1).write.format("parquet").mode("overwrite").saveAsTable("dimLock_parquet")

In [7]:
spark.stop()